# Module 4: Bulletproof Infrastructure Code (Exceptions & Context Managers)

**Concepts:** `try/except` · raising errors · custom exceptions · `with` / context managers

**Audience:** Automation must **fail loudly** with context — never swallow errors on prod changes.

---

**Rhythm:** markdown theory → runnable example.


## 1. Why exceptions beat return codes everywhere

Returning `-1` or `None` on failure forces every caller to remember to check.

**Exceptions** unwind the stack until something handles them — good for rare failure paths in deploy scripts.


In [ ]:
def parse_port(text: str) -> int:
    port = int(text)
    if not 1 <= port <= 65535:
        raise ValueError(f"port out of range: {port}")
    return port


try:
    print(parse_port("6443"))
    print(parse_port("99999"))
except ValueError as exc:
    print("handled:", exc)


## 2. `try` / `except` / `else` / `finally`

- **except** — handle expected failures
- **else** — runs if no exception (optional)
- **finally** — always runs (cleanup)


In [ ]:
def load_config(name: str) -> dict:
    if name == "missing":
        raise FileNotFoundError(name)
    return {"replicas": 3}


cfg = None
try:
    cfg = load_config("app.yaml")
except FileNotFoundError as exc:
    print("using defaults; missing", exc)
else:
    print("loaded", cfg)
finally:
    print("config step finished")


## 3. Catch specific exceptions

Avoid bare `except:` — you hide bugs. Catch `TimeoutError`, `ValueError`, etc.


In [ ]:
def divide(a: float, b: float) -> float:
    try:
        return a / b
    except ZeroDivisionError:
        raise ValueError("cannot divide by zero in metric ratio") from None


try:
    divide(1, 0)
except ValueError as e:
    print(e)


## 4. Custom exception hierarchy

**Platform:** `InfraError` → `NetworkError`, `ConfigError` lets callers handle groups.


In [ ]:
class InfraError(Exception):
    pass


class NetworkError(InfraError):
    pass


class ConfigError(InfraError):
    pass


def connect(host: str) -> None:
    if host == "0.0.0.0":
        raise NetworkError(f"refusing bogus host {host}")


try:
    connect("0.0.0.0")
except InfraError as exc:
    print(type(exc).__name__, exc)


## 5. Context managers and `with`

`with` guarantees setup/teardown (close file, release lock) even on crash.

Files are the classic case (Module 5 expands `pathlib`).


In [ ]:
from contextlib import contextmanager


@contextmanager
def maintenance_window(name: str):
    print(f"ENTER maintenance {name}")
    try:
        yield
    finally:
        print(f"EXIT maintenance {name}")


with maintenance_window("ncs57-upgrade"):
    print("  patching node")


## 6. Class-based context manager (`__enter__` / `__exit__`)

Same idea as `@contextmanager`, explicit for connection objects.


In [ ]:
class FakeSession:
    def __init__(self, target: str):
        self.target = target
        self.open = False

    def __enter__(self):
        self.open = True
        print("session open", self.target)
        return self

    def __exit__(self, exc_type, exc, tb):
        self.open = False
        print("session closed", self.target)
        return False  # do not suppress exceptions


with FakeSession("vault") as s:
    print("  token lookup", s.open)


## 7. Capstone — domain errors + `with`

Simulate API client that always closes session.


In [ ]:
class ApiError(InfraError):
    pass


class ApiClient:
    def __init__(self, base: str):
        self.base = base

    def __enter__(self):
        print("connect", self.base)
        return self

    def __exit__(self, *args):
        print("disconnect", self.base)
        return False

    def get(self, path: str) -> str:
        if path == "/fail":
            raise ApiError("upstream 503")
        return "ok"


with ApiClient("https://gitlab.example") as client:
    print(client.get("/health"))


---
# Exercises


**Ex 1** — Function raising `ConfigError` if required key missing from dict.


In [ ]:
cfg = {"host": "x"}
# TODO: require key "port"


**Ex 2** — `try/except` around `int('not-a-port')` print friendly message.


In [ ]:
# TODO


**Ex 3** — `@contextmanager` `timer(name)` printing elapsed using `time.perf_counter()`.


In [ ]:
# TODO


**Ex 4** — Custom `RollbackError(InfraError)`; function `deploy()` raises it; catch as `InfraError`.


In [ ]:
# TODO


---
# Solutions


In [ ]:
class ConfigError(InfraError):
    pass


def require_key(cfg: dict, key: str) -> str:
    if key not in cfg:
        raise ConfigError(f"missing {key}")
    return cfg[key]


try:
    require_key({"host": "x"}, "port")
except ConfigError as e:
    print(e)


In [ ]:
try:
    int("not-a-port")
except ValueError:
    print("invalid port literal")


In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def timer(name: str):
    start = time.perf_counter()
    yield
    print(f"{name}: {time.perf_counter() - start:.3f}s")


with timer("step"):
    time.sleep(0.05)


In [ ]:
class RollbackError(InfraError):
    pass


def deploy() -> None:
    raise RollbackError("helm rollback required")


try:
    deploy()
except InfraError as e:
    print("caught", e)


**Next:** Module 5 — files, regex, logging, subprocess.
